# Noisy Tree Encoding — does margin noise cure the overfitting?

**The idea (professor's suggestion):** the overfitting comes from the tree-encoding features, so inject **Gaussian noise** into them and squash with a **sigmoid** so they stay in (0,1), close to their actual values — a regularizer aimed at the exact features that memorize.

**How it's implemented (the version that can actually work):** noise is added to the **margin** (distance to each split's threshold) *before* the sigmoid, and is **resampled fresh on every training batch** — like dropout, not a one-time corrupted dataset:

```
bit = sigmoid( (threshold − x  +  σ·ε) / τ )      ε ~ N(0,1), new every batch
```
- bits **near a split boundary** (the fingerprint-y, memorizable ones) flip often
- bits **far from the boundary** stay solidly ≈0/1 — 'close to the actual values'
- **evaluation always uses the clean encoding** (σ=0) — noise is train-only

### The experiment (credit dataset, x+tree, OOB-honest encoding)
Five arms, identical in every other way (dropout 0, L1 0 — so noise is the *only* variable):

| arm | τ (softness) | σ (noise) | question it answers |
|---|---|---|---|
| hard control | 0 | 0 | the baseline everyone has |
| soft control | 0.1 | 0 | does softness alone help? |
| light noise | 0.1 | 0.1 | |
| medium noise | 0.1 | 0.2 | the professor's idea, three strengths |
| strong noise | 0.1 | 0.4 | |

**What success looks like:** train AUC stops snapping to 1.0, and test AUC for some σ beats both controls. Ceiling to beat: LightGBM ≈ **0.8453**; previous best x+tree ≈ **0.8477**.

⏱ ~30–40 min on an A100 (5 arms × 2-member ensembles × 400 epochs). Runtime → GPU, then Run all.

In [ ]:
# 1 · GPU check
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -> Runtime > GPU')

In [ ]:
# 2 · get the code
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull

In [ ]:
# 3 · install deps
!pip install -q openml catboost optuna

In [ ]:
# 3b · OPTIONAL — only if OpenML 504s: upload openml_cache_clean5.tar.gz (else press Cancel)
import os, glob, tarfile
dst = '/root/.cache/openml/org/openml/www'
if glob.glob(dst + '/tasks/361055'):
    print('credit already cached — skip')
else:
    try:
        from google.colab import files; files.upload()
    except Exception as e: print('skipped:', e)
    hits = glob.glob('/content/**/openml_cache_clean5.tar.gz', recursive=True)
    if hits:
        os.makedirs(dst, exist_ok=True)
        with tarfile.open(hits[0]) as t: t.extractall(dst)
        print('cache extracted')
    else: print('no bundle — will use OpenML directly')

In [ ]:
# 4 · THE SWEEP — five arms, only (tau, sigma) changes
base = ('--task 361055 --views x+tree --encoding oob --ensemble 2 --epochs 400 '
        '--dropout 0 --l1 0 --weight-decay 1e-3 --lr 3e-4 --batch-size 128 --device auto')
ARMS = [('hard_ctrl', 0.0, 0.0), ('soft_ctrl', 0.1, 0.0),
        ('noise_010', 0.1, 0.1), ('noise_020', 0.1, 0.2), ('noise_040', 0.1, 0.4)]
for name, tau, sig in ARMS:
    print(f'\n############  {name}: tau={tau} sigma={sig}  ############')
    !python -u run_fusion.py {base} --tau {tau} --enc-noise {sig} --out results/fusion/noise_{name}

In [ ]:
# 5 · summary table: test AUC + memorization gauge per arm
import json, glob, pandas as pd
rows = []
for d in sorted(glob.glob('results/fusion/noise_*')):
    s = json.load(open(glob.glob(d + '/fusion_*.json')[0]))
    e = pd.read_csv(glob.glob(d + '/fusion_*_epochs.csv')[0])
    r = s['results'][0]
    rows.append(dict(arm=d.split('noise_')[-1], tau=s['tau'], sigma=s.get('enc_noise', 0),
                     test_auc=r['test_auc'], best_val=r['best_val_auc'],
                     final_train_auc=e.groupby('epoch').train_auc.mean().iloc[-1],
                     ceiling=s['tree_ceiling']))
t = pd.DataFrame(rows).sort_values('sigma')
print(t.round(4).to_string(index=False))
best = t.loc[t.test_auc.idxmax()]
print(f"\nbest arm: {best.arm} (test {best.test_auc:.4f})  vs hard control "
      f"{t[t.arm=='hard_ctrl'].test_auc.iloc[0]:.4f}  vs tree ceiling {best.ceiling:.4f}")
print('memorization gauge: final_train_auc near 1.0 = still memorizing; lower = noise is working')

In [ ]:
# 6 · (optional) does the winner stack with dropout? set SIGMA to the best arm's sigma
SIGMA = 0.2
!python -u run_fusion.py {base} --tau 0.1 --enc-noise {SIGMA} --dropout 0.3 \
    --out results/fusion/noise_plus_dropout

In [ ]:
# 7 · show every figure
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/fusion/noise_*/*.png')):
    print('==', f, '==')
    display(Image(f))

In [ ]:
# 8 · download everything
import shutil
from google.colab import files
shutil.make_archive('noisy_encoding_credit', 'zip', 'results/fusion')
files.download('noisy_encoding_credit.zip')